# What separates casual riders from members?

Goal: train an MLP (TensorFlow/Keras) to predict `member_casual`, explain it with SHAP, and profile the
casual riders the model sees as "almost members" (conversion targets).

Steps: features → baseline vs MLP → leakage check → SHAP → "almost member" casuals.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import polars as pl
import shap
from tensorflow import keras
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
N_SAMPLE = None
keras.utils.set_random_seed(SEED)

/Volumes/PortableSSD/Projetos/google-analytics-capstone-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data and features
Cyclical variables (hour, weekday, month) used sin/cos; proxy-prone columns were kept separate for the leakage check.

In [2]:
df = pl.read_parquet("../data/processed/clean_tripdata.parquet").filter(
    pl.col("start_inside_chicago") & pl.col("start_lat").is_not_null()
)
if N_SAMPLE:
    df = df.sample(n=N_SAMPLE, seed=SEED)  # random sample keeps the class ratio
df["member_casual"].value_counts(normalize=True)

member_casual,proportion
str,f64
"""casual""",0.350268
"""member""",0.649732


In [3]:
def cyc(col: str, period: int) -> list[pl.Expr]:
    angle = pl.col(col) * (2 * np.pi / period)
    return [angle.sin().alias(f"{col}_sin"), angle.cos().alias(f"{col}_cos")]


X = df.select(
    *cyc("start_hour", 24),
    *cyc("day_of_week", 7),
    *cyc("month", 12),
    pl.col("is_weekend").cast(pl.Float32),
    # Clips rides between 0 and 6 hours and applies log(x + 1)
    pl.col("ride_length").clip(0, 6 * 3600).log1p().alias("log_ride_length"),
    # Clips rides between 0 and 30 km
    pl.col("distance_km").clip(0, 30),
    pl.col("is_round_trip").fill_null(False).cast(pl.Float32),
    (pl.col("rideable_type") == "electric_bike").cast(pl.Float32).alias("is_ebike"),
    (pl.col("rideable_type") == "classic_bike").cast(pl.Float32).alias("is_classic"),
    "start_lat",
    "start_lng",
    pl.col("has_start_station").cast(pl.Float32),
    pl.col("has_end_station").cast(pl.Float32),
).fill_null(0.0)

y = (df["member_casual"] == "member").to_numpy().astype(np.float32)
names = X.columns
Xn = X.to_numpy().astype(np.float32)

idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)
tr, va = train_test_split(tr, test_size=0.125, stratify=y[tr], random_state=SEED)

mu, sd = Xn[tr].mean(0), Xn[tr].std(0) + 1e-6  # stats from train only
Z = (Xn - mu) / sd

## Baselines (gradient boosting, XGBoost) vs MLP

Note: HistGradientBoostingClassifier was chosen because it performs better on bigger datasets than GradientBoostingClassifier. It uses a histogram-based approach.

In [4]:
gb = HistGradientBoostingClassifier(max_iter=300, random_state=SEED).fit(Z[tr], y[tr])
p_gb = gb.predict_proba(Z[te])[:, 1]
print(f"Gradient Boosting: AUC = {roc_auc_score(y[te], p_gb):.4f} | AP = {average_precision_score(y[te], p_gb):.4f}")

Gradient Boosting: AUC = 0.7451 | AP = 0.8301


In [5]:
xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="auc",
    early_stopping_rounds=30,
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)
xgb.fit(Z[tr], y[tr], eval_set=[(Z[va], y[va])], verbose=False)
p_xgb = xgb.predict_proba(Z[te])[:, 1]
print(f"XGBoost: AUC = {roc_auc_score(y[te], p_xgb):.4f} | AP = {average_precision_score(y[te], p_xgb):.4f} | trees = {xgb.best_iteration + 1}")

XGBoost: AUC = 0.7514 | AP = 0.8351 | trees=1000


In [6]:
def build_network(d: int) -> keras.Model:
    return keras.Sequential([
        keras.layers.Input((d,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation="sigmoid"),
    ])

def fit(cols: list[int], epochs: int = 15) -> keras.Model:
    m = build_network(len(cols))
    m.compile(
        keras.optimizers.AdamW(2e-3, weight_decay=1e-4),
        "binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],
    )
    m.fit(
        Z[tr][:, cols], y[tr],
        validation_data=(Z[va][:, cols], y[va]),
        epochs=epochs,
        batch_size=1024,
        verbose=0,
        callbacks=[keras.callbacks.EarlyStopping("val_auc", mode="max", patience=3,
                                                 restore_best_weights=True)],
    )
    return m

def predict(m: keras.Model, cols: list[int]) -> np.ndarray:
    return m.predict(Z[te][:, cols], batch_size=8192, verbose=0).ravel()

all_cols = list(range(len(names)))
mlp = fit(all_cols)
p_mlp = predict(mlp, all_cols)
print(f"MLP: AUC = {roc_auc_score(y[te], p_mlp):.4f} | AP = {average_precision_score(y[te], p_mlp):.4f}")

MLP: AUC = 0.7297 | AP = 0.8191


## Leakage check
`rideable_type` and `has_*_station` may reflect how the data was recorded rather than rider behavior. To test so, a new MLP was trained without those features.

Result: since AUC barely dropped without them, it is possible to be confident that the conclusions are behavioral. If it had dropped substantially, their importance should be treated with caution.

In [7]:
PROXIES = ["is_ebike", "is_classic", "has_start_station", "has_end_station"]
keep = [i for i, n in enumerate(names) if n not in PROXIES]
mlp_np = fit(keep)
print(f"MLP (without proxies): AUC = {roc_auc_score(y[te], predict(mlp_np, keep)):.4f}")

MLP (without proxies): AUC = 0.7135


## SHAP (global importance)
Model-agnostic `PermutationExplainer` over the Keras model. Cyclical sin/cos pairs were summed back into one feature.

In [35]:
import plotly.graph_objects as go

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, 'Segoe UI', sans-serif"
LOW = "#c9c7bf"
FEATURE_LABEL = {
    "start_hour": "Start hour", "day_of_week": "Day of week", "month": "Month", "is_weekend": "Weekend",
    "log_ride_length": "Ride length", "distance_km": "Distance", "is_round_trip": "Round trip",
    "is_ebike": "E-bike", "is_classic": "Classic bike", "start_lat": "Start latitude",
    "start_lng": "Start longitude", "has_start_station": "Has start station", "has_end_station": "Has end station",
}

bg = shap.sample(Z[tr], 100, random_state=SEED)
explainer = shap.PermutationExplainer(
    lambda a: mlp.predict(a, batch_size=8192, verbose=0).ravel(), bg, silent=True
)
sidx = np.random.RandomState(SEED).choice(len(te), 500, replace=False)
sv = explainer(Z[te][sidx], max_evals=2 * len(names) + 1, silent=True).values  # (500, n_features)

def base(n: str) -> str:
    return n[:-4] if n.endswith(("_sin", "_cos")) else n

groups: dict[str, list[int]] = {}
for i, n in enumerate(names):
    groups.setdefault(base(n), []).append(i)

def group_importance(values: np.ndarray) -> pl.DataFrame:
    # sum signed contributions inside each group, then take the mean absolute value
    return (
        pl.DataFrame({g: np.abs(values[:, ix].sum(1)) for g, ix in groups.items()})
        .mean().transpose(include_header=True, header_name="feature", column_names=["mean_abs_shap"])
        .sort("mean_abs_shap")
    )

imp = group_importance(sv)  # ascending

# Headline numbers
vals = imp["mean_abs_shap"].to_list()
feats = imp["feature"].to_list()
lab = [FEATURE_LABEL.get(f, f) for f in feats]
top2_share = sum(vals[-2:]) / sum(vals) * 100

fig = go.Figure(
    go.Bar(
        y=lab, x=vals, orientation="h", width=0.7,
        marker=dict(color=[INK_2 if i >= len(vals) - 2 else LOW for i in range(len(vals))],
                    line=dict(width=2, color=SURFACE)),
        text=[f"{v:.3f}" for v in vals], textposition="outside", cliponaxis=False,
        textfont=dict(size=12, color=INK),
        hovertemplate="<b>%{y}</b><br>mean |SHAP| = %{x:.3f}<extra></extra>",
    )
)
fig.update_layout(
    title=dict(
        text=(f"<b>{lab[-1]} and {lab[-2].lower()} carry {top2_share:.0f}% of what the model uses "
              "to tell members from casual riders</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Mean |SHAP| per feature, in change of P(member). 500 test rides; sin/cos pairs combined</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family=FONT, color=INK_2),
    margin=dict(t=110, l=20, r=40, b=30), height=480, showlegend=False,
)
fig.update_xaxes(visible=False, range=[0, max(vals) * 1.18])
fig.update_yaxes(showgrid=False, ticks="", showline=False, automargin=True, tickfont=dict(size=13, color=INK))
fig.show()
fig.write_image("../figs/09_shap_importance.svg", width=1000)

### XGBoost: exact SHAP with `TreeExplainer`

Same 500 test rides as the MLP.

Since XGBoost SHAP values are in log-odds and the MLP's are in probability, the comparison used each feature's **share of total importance** (%) instead of raw values.

In [36]:
import plotly.graph_objects as go

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, 'Segoe UI', sans-serif"
MODEL_COLOR = {"MLP": INK_2, "XGBoost": "#a8a69e"}

sv_xgb = shap.TreeExplainer(xgb).shap_values(Z[te][sidx])  # exact, fast (tree-specific)
imp_xgb = group_importance(sv_xgb)

cmp = (
    imp.rename({"mean_abs_shap": "MLP"})
    .join(imp_xgb.rename({"mean_abs_shap": "XGBoost"}), on="feature", how="left")
    .with_columns((pl.col(c) / pl.col(c).sum() * 100) for c in ("MLP", "XGBoost"))
)  # ordered by MLP importance, ascending
rho = cmp.select(pl.corr("MLP", "XGBoost", method="spearman")).item()
verdict = "agree" if rho >= 0.7 else "only partly agree"
lab = [FEATURE_LABEL.get(f, f) for f in cmp["feature"]]

fig = go.Figure()
for m in ["XGBoost", "MLP"]:  # first trace draws lower in each group, so MLP ends on top
    v = cmp[m].to_list()
    fig.add_trace(
        go.Bar(
            y=lab, x=v, name=m, orientation="h",
            marker=dict(color=MODEL_COLOR[m], line=dict(width=2, color=SURFACE)),
            text=[f"{x:.1f}%" for x in v], textposition="outside", cliponaxis=False,
            textfont=dict(size=11, color=INK),
            hovertemplate=f"<b>%{{y}}</b><br>{m}: %{{x:.1f}}% of total |SHAP|<extra></extra>",
        )
    )
fig.update_layout(
    barmode="group", bargap=0.25,
    title=dict(
        text=(f"<b>MLP and XGBoost {verdict} on what matters (rank correlation {rho:.2f})</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Share of total |SHAP| by feature. Same 500 test rides; XGBoost uses exact TreeExplainer</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family=FONT, color=INK_2),
    legend=dict(orientation="h", x=0, y=-0.04, traceorder="reversed", font=dict(size=12, color=INK_2)),
    margin=dict(t=110, l=20, r=40, b=50), height=620,
)
fig.update_xaxes(visible=False, range=[0, max(cmp["MLP"].max(), cmp["XGBoost"].max()) * 1.18])
fig.update_yaxes(showgrid=False, ticks="", showline=False, automargin=True, tickfont=dict(size=13, color=INK))
fig.show()
fig.write_image("../figs/11_shap_mlp_vs_xgboost.svg", width=1000)

## "Almost member" casuals
Among real casual rides in the test set, it was compared those the model scores as likely members (P ≥ 0.7) with the typical casual (P < 0.3). This gap shows which behaviors separate conversion targets.

In [10]:
casual = y[te] == 0
p_c = p_mlp[casual]
tdf = df[te.tolist()].filter(pl.Series(casual))
tdf = tdf.with_columns(pl.Series("p_member", p_c))
tdf = tdf.with_columns(
    pl.when(pl.col("p_member") >= 0.7).then(pl.lit("almost member"))
    .when(pl.col("p_member") < 0.3).then(pl.lit("typical casual"))
    .alias("segment")
).filter(pl.col("segment").is_not_null())

profile = tdf.group_by("segment").agg(
    pl.len().alias("rides"),
    (pl.col("ride_length").median() / 60).round(1).alias("median_min"),
    pl.col("distance_km").median().round(2).alias("median_km"),
    (pl.col("is_weekend").mean() * 100).round(1).alias("pct_weekend"),
    (pl.col("is_round_trip").mean() * 100).round(1).alias("pct_round_trip"),
    ((pl.col("rideable_type") == "electric_bike").mean() * 100).round(1).alias("pct_ebike"),
    (pl.col("start_hour").is_between(7, 9) | pl.col("start_hour").is_between(16, 18)).mean().mul(100).round(1).alias("pct_commute_hours"),
)
profile

segment,rides,median_min,median_km,pct_weekend,pct_round_trip,pct_ebike,pct_commute_hours
str,u32,f64,f64,f64,f64,f64,f64
"""typical casual""",57327,29.6,1.52,59.7,21.3,62.5,22.7
"""almost member""",106054,7.8,1.6,12.4,0.3,64.9,52.8


In [37]:
from plotly.subplots import make_subplots

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, 'Segoe UI', sans-serif"
GRID = "#e1e0d9"
SEGMENT = {  # red = casual behaviour, blue = member behaviour (same coding as share)
    "typical casual": ("Typical casual", "#e34948"),
    "almost member": ("Almost member", "#2a78d6"),
}
COMMUTE = [7, 8, 9, 16, 17, 18]

agg = {
    row["segment"]: row
    for row in tdf.group_by("segment").agg(
        (pl.col("is_weekend").mean() * 100).alias("Weekend rides"),
        (pl.col("start_hour").is_in(COMMUTE).mean() * 100).alias("Commute-hour rides"),
        (pl.col("is_round_trip").mean() * 100).alias("Round trips"),
        ((pl.col("rideable_type") == "electric_bike").mean() * 100).alias("E-bike rides"),
        (pl.col("ride_length").median() / 60).alias("Median ride length"),
    ).iter_rows(named=True)
}
PCT_METRICS = ["Weekend rides", "Commute-hour rides", "Round trips", "E-bike rides"]
gap = {m: abs(agg["almost member"][m] - agg["typical casual"][m]) for m in PCT_METRICS}
order = sorted(PCT_METRICS, key=gap.get)  # ascending: the largest gap ends on top
top = order[-1]

# One column, two stacked sections (different units, so each gets its own header and x scale)
fig = make_subplots(
    rows=2, cols=1, row_heights=[0.8, 0.2], vertical_spacing=0.16,
    subplot_titles=["% of each segment's rides, sorted by gap", "Median ride length"],
)
fig.update_annotations(font=dict(size=13, color=INK_2), x=0, xanchor="left")  # left-aligned section headers

def dumbbell(metrics, row, fmt, first_legend):
    for m in metrics:
        a, t = agg["almost member"][m], agg["typical casual"][m]
        fig.add_trace(go.Scatter(  # connector between the two segments
            x=[a, t], y=[m, m], mode="lines", showlegend=False, hoverinfo="skip",
            line=dict(color=GRID, width=4),
        ), row=row, col=1)
        for s, (name, color) in SEGMENT.items():
            v = agg[s][m]
            right = v >= max(a, t)  # label the higher dot on its right, the lower on its left
            fig.add_trace(go.Scatter(
                x=[v], y=[m], mode="markers+text", name=name, legendgroup=s,
                showlegend=first_legend and m == metrics[0],
                marker=dict(color=color, size=13, line=dict(width=2, color=SURFACE)),
                text=[f"<b>{fmt(v)}</b>"], textposition="middle right" if right else "middle left",
                textfont=dict(size=13, color=INK), cliponaxis=False,
                hovertemplate=f"<b>{name}</b><br>{m}: {fmt(v)}<extra></extra>",
            ), row=row, col=1)

dumbbell(order, 1, lambda v: f"{v:.0f}%" if v >= 1 else f"{v:.1f}%", first_legend=True)
dumbbell(["Median ride length"], 2, lambda v: f"{v:.0f} min", first_legend=False)

a, t = agg["almost member"][top], agg["typical casual"][top]
fig.update_layout(
    title=dict(
        text=(f"<b>'Almost member' casuals differ most in {top.lower()}: {a:.0f}% versus {t:.0f}%</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Gap sizes are in percentage points; ride length is shown separately, in minutes</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family=FONT, color=INK_2),
    legend=dict(orientation="h", x=0, y=-0.08, font=dict(size=12, color=INK_2)),
    margin=dict(t=120, l=20, r=40, b=110), height=560,
    annotations=list(fig.layout.annotations) + [dict(
        x=0, y=-0.2, xref="paper", yref="paper", showarrow=False, xanchor="left",
        font=dict(size=11, color=MUTED),
        text=("Casual rides only. 'Almost member': MLP scores P(member) ≥ 0.7; 'typical casual': P(member) &lt; 0.3. "
              "Round trips exclude rides where it can't be determined."),
    )],
)
fig.update_xaxes(visible=False)
fig.update_xaxes(range=[-16, 112], row=1, col=1)  # room for the label left of a ~0% dot
fig.update_xaxes(range=[-16 / 112 * 40, 40], row=2, col=1)  # minutes; same left padding as the % section
fig.update_yaxes(showgrid=False, ticks="", showline=False, tickfont=dict(size=13, color=INK))
fig.update_yaxes(visible=False, row=2, col=1)  # the section header already names this row
fig.update_yaxes(categoryorder="array", categoryarray=order, row=1, col=1)
fig.show()
fig.write_image("../figs/12_almost_member_gaps.svg", width=1000)

In [38]:
import plotly.graph_objects as go

SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
FONT = "system-ui, -apple-system, 'Segoe UI', sans-serif"
GRID = "#e1e0d9"
SEGMENT = {
    "typical casual": ("Typical casual", "#e34948"),
    "almost member": ("Almost member", "#2a78d6"),
}
COMMUTE = [7, 8, 9, 16, 17, 18]

hour = (
    tdf.group_by("segment", "start_hour").agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum().over("segment") * 100).alias("pct"))
)
pct = {(s, h): p for s, h, p in hour.select("segment", "start_hour", "pct").iter_rows()}
series = {s: [pct.get((s, h), 0.0) for h in range(24)] for s in SEGMENT}
commute = {s: sum(series[s][h] for h in COMMUTE) for s in SEGMENT}

fig = go.Figure()
for s, (name, color) in SEGMENT.items():
    y = series[s]
    fig.add_trace(
        go.Scatter(
            x=list(range(24)), y=y, name=name, mode="lines",
            line=dict(color=color, width=2.5),
            hovertemplate="%{y:.1f}% of rides<extra>" + name + "</extra>",
        )
    )
# Label each line at its end. If the two ends are close, nudge the labels apart so they don't collide.
close = abs(series["almost member"][-1] - series["typical casual"][-1]) < 1.5
lower = min(SEGMENT, key=lambda s: series[s][-1])
for s, (name, color) in SEGMENT.items():
    fig.add_annotation(
        x=23, y=series[s][-1], xanchor="left", xshift=10, showarrow=False,
        yshift=(-9 if s == lower else 9) if close else 0,
        text=f"<b>{name}</b>", font=dict(size=13, color=INK),
    )
fig.update_layout(
    hovermode="x unified",
    title=dict(
        text=(f"<b>{commute['almost member']:.0f}% of 'almost member' casual rides start at commute hours, "
              f"versus {commute['typical casual']:.0f}% of typical casual rides</b>"
              f"<br><span style='font-size:13px;color:{INK_2}'>"
              "Start hour as % of each segment's rides. Commute hours = 7–9h and 16–18h</span>"),
        x=0, xanchor="left", font=dict(size=18, color=INK),
    ),
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(family=FONT, color=INK_2),
    legend=dict(orientation="h", x=0, y=-0.15, font=dict(size=12, color=INK_2)),
    margin=dict(t=110, l=50, r=130, b=100), height=480,
)
fig.add_annotation(
    x=0, y=-0.34, xref="paper", yref="paper", showarrow=False, xanchor="left",
    font=dict(size=11, color=MUTED),
    text="Casual rides only. 'Almost member': MLP scores P(member) ≥ 0.7; 'typical casual': P(member) &lt; 0.3.",
)
fig.update_yaxes(rangemode="tozero", ticksuffix="%", showline=False, ticks="",
                 gridcolor=GRID, gridwidth=1, zeroline=False, tickfont=dict(size=12, color=MUTED))
fig.update_xaxes(tickmode="array", tickvals=list(range(0, 24, 3)), ticktext=[f"{h}h" for h in range(0, 24, 3)],
                 showgrid=False, showline=True, linecolor=GRID, ticks="", tickfont=dict(size=12, color=INK_2))
# Highlight: shaded commute-hour bands, labeled above the plot area
for a, b in [(7, 9), (16, 18)]:
    fig.add_vrect(x0=a - 0.5, x1=b + 0.5, fillcolor=GRID, opacity=0.5, line_width=0, layer="below")
    fig.add_annotation(x=(a + b) / 2, y=1, xref="x", yref="paper", xanchor="center", yanchor="bottom",
                       showarrow=False, text="Commute hours", font=dict(size=12, color=INK_2))
fig.show()
fig.write_image("../figs/10_almost_member_hours.svg", width=1000)